<a href="https://colab.research.google.com/github/IrenaSpivak15/IrenaSpivak15/blob/claude%2Fphoto-avatar-7nfmuj/01_medical_invoice_spend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">

# סיפור לקוח 1 · רפואה: איפה הולך תקציב הרכש

**תרחיש לדוגמה. הנתונים במחברת סינתטיים.**

**הלקוח:** רשת מרפאות פרטיות עם ארבעה סניפים. ההכנסות יציבות, הרווחיות נשחקת, ואף אחד לא יודע להצביע על הסיבה.

**הכאב:** אלפי שורות חשבונית מספקים בטקסט חופשי ולא אחיד. אותו פריט נכתב כ-`nitrile gloves size L box of 100` בסניף אחד וכ-`glove nitril lrg bx100` בסניף אחר. אין קטגוריות, ולכן אין השוואה.

**איפה הכסף הסתתר:** סניף אחד משלם על ציוד מגן יותר מהסניפים האחרים, על אותם פריטים בדיוק. בדוח החודשי זה נבלע בשורת "הוצאות תפעול".

**מה בנינו:**
1. טוקניזציה WordPiece עם אוצר המילים של BERT (קובץ ה-Vocab מהקורס, ‏30,522 טוקנים)
2. המרת כל שורה לטנזור של מזהי טוקנים
3. מסווג `EmbeddingBag` + MLP שמשייך כל שורה לקטגוריית הוצאה
4. פילוח הוצאות לפי סניף וקטגוריה, ואיתור תשלומי יתר לפי מק״ט

**חוסן עסקי:** שליטה בעלויות הרכש היא מרווח הביטחון של המרפאה כשתשלומים מקופות החולים ומחברות הביטוח מתעכבים.

**מה זה משלים מהחומר:** Vocab ו-WordPiece, טנזורים ו-offsets, ‏ReLU ו-Dropout.

</div>

<div dir="rtl">

## 0. הכנה

</div>

In [1]:
import re, random
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn

torch.manual_seed(42); random.seed(42); np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


<div dir="rtl">

## 1. טוענים את אוצר המילים

הקובץ הוא `vocab.txt` של `bert-base-uncased`: טוקן אחד בכל שורה, ומספר השורה הוא מזהה הטוקן.
הקובץ מהקורס יוצא מ-Google Docs, ולכן יש בו לוכסנים לפני תווים כמו `[` ו-`#`. הפונקציה מנקה אותם.

</div>

In [4]:
import urllib.request

# Download the official bert-base-uncased vocabulary file
vocab_url = "https://huggingface.co/bert-base-uncased/resolve/main/vocab.txt"
vocab_file = "vocab.txt"

try:
    print(f"Downloading {vocab_url}...")
    urllib.request.urlretrieve(vocab_url, vocab_file)
    print("Successfully downloaded vocab.txt!")
except Exception as e:
    print(f"Error downloading file: {e}")

Successfully downloaded vocab.txt!


In [5]:
VOCAB_PATH = Path("vocab.txt")  # שמרו את קובץ ה-Vocab מהקורס בשם הזה, ליד המחברת

def load_vocab(path):
    lines = path.read_text(encoding="utf-8").split("\n")
    vocab = {}
    for i, line in enumerate(lines):
        tok = re.sub(r"\\(.)", r"\1", line.strip())  # \[PAD\] -> [PAD], \#\#s -> ##s
        if tok and tok not in vocab:
            vocab[tok] = i
    return vocab

vocab = load_vocab(VOCAB_PATH)
print(len(vocab), "tokens")
print({t: vocab[t] for t in ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "the", "glove", "##s"]})

30522 tokens
{'[PAD]': 0, '[UNK]': 100, '[CLS]': 101, '[SEP]': 102, 'the': 1996, 'glove': 15913, '##s': 2015}


<div dir="rtl">

## 2. WordPiece בעבודת יד

האלגוריתם חמדן: לוקחים את הרצף הארוך ביותר שקיים באוצר המילים, ומה שנשאר ממשיך עם תחילית `##`.
כך גם מילה עם שגיאת כתיב (`nitril`) מתפרקת לחלקים מוכרים, במקום להפוך ל-`[UNK]`.

</div>

In [6]:
def basic_tokenize(text):
    return re.findall(r"[a-z]+|\d+|[^\sa-z\d]", text.lower())

def wordpiece(word):
    pieces, start = [], 0
    while start < len(word):
        end, cur = len(word), None
        while start < end:
            sub = word[start:end] if start == 0 else "##" + word[start:end]
            if sub in vocab:
                cur = sub
                break
            end -= 1
        if cur is None:
            return ["[UNK]"]
        pieces.append(cur)
        start = end
    return pieces

def tokenize(text):
    return [p for w in basic_tokenize(text) for p in wordpiece(w)]

def encode(text):
    return [vocab.get(t, vocab["[UNK]"]) for t in tokenize(text)]

for s in ["nitrile gloves size L box of 100", "glove nitril lrg bx100", "lidocaine 1% vial"]:
    print(f"{s!r}\n  tokens: {tokenize(s)}\n  ids:    {encode(s)}\n")

'nitrile gloves size L box of 100'
  tokens: ['ni', '##tri', '##le', 'gloves', 'size', 'l', 'box', 'of', '100']
  ids:    [9152, 18886, 2571, 11875, 2946, 1048, 3482, 1997, 2531]

'glove nitril lrg bx100'
  tokens: ['glove', 'ni', '##tri', '##l', 'l', '##rg', 'b', '##x', '100']
  ids:    [15913, 9152, 18886, 2140, 1048, 10623, 1038, 2595, 2531]

'lidocaine 1% vial'
  tokens: ['lid', '##oca', '##ine', '1', '%', 'vial']
  ids:    [11876, 24755, 3170, 1015, 1003, 28475]



In [12]:
# בדיקת לוגיקת ה-WordPiece על מגוון קלטים שונים
test_phrases = [
    "Standard medical gloves",
    "nitril glovezzz typo test",
    "100% sterile syringe",
    "nonexistentword1234",
    "Lidocaine 2% with epinephrine"
]

print("=== WordPiece Tokenization Test ===\n")
for phrase in test_phrases:
    tokens = tokenize(phrase)
    token_ids = encode(phrase)
    print(f"Original: {phrase!r}")
    print(f"Tokens:   {tokens}")
    print(f"IDs:      {token_ids}")
    print("-" * 40)

=== WordPiece Tokenization Test ===

Original: 'Standard medical gloves'
Tokens:   ['standard', 'medical', 'gloves']
IDs:      [3115, 2966, 11875]
----------------------------------------
Original: 'nitril glovezzz typo test'
Tokens:   ['ni', '##tri', '##l', 'glove', '##zz', '##z', 'ty', '##po', 'test']
IDs:      [9152, 18886, 2140, 15913, 13213, 2480, 5939, 6873, 3231]
----------------------------------------
Original: '100% sterile syringe'
Tokens:   ['100', '%', 'sterile', 'sy', '##ring', '##e']
IDs:      [2531, 1003, 25403, 25353, 4892, 2063]
----------------------------------------
Original: 'nonexistentword1234'
Tokens:   ['none', '##xi', '##sten', '##t', '##word', '123', '##4']
IDs:      [3904, 9048, 16173, 2102, 18351, 13138, 2549]
----------------------------------------
Original: 'Lidocaine 2% with epinephrine'
Tokens:   ['lid', '##oca', '##ine', '2', '%', 'with', 'ep', '##ine', '##ph', '##rine']
IDs:      [11876, 24755, 3170, 1016, 1003, 2007, 4958, 3170, 8458, 11467]
------

<div dir="rtl">

## 3. נתוני חשבוניות סינתטיים

שש קטגוריות, ארבעה סניפים, ניסוחים לא אחידים ושגיאות כתיב.
**שתלנו בכוונה פער:** סניף חיפה משלם 25% יותר על ציוד מגן (PPE). המטרה: לבדוק שהשיטה מוצאת אותו.

</div>

In [7]:
CATALOG = {
    "PPE":      [("nitrile gloves size {s} box of 100", 38), ("surgical mask type iir box 50", 22), ("isolation gown disposable", 9)],
    "Syringes": [("syringe 5ml luer lock x100", 31), ("hypodermic needle 23g box 100", 17), ("insulin syringe 1ml x100", 29)],
    "Lab":      [("blood collection tube edta 4ml x100", 45), ("glucose test strips x50", 26), ("crp reagent kit", 140)],
    "Cleaning": [("disinfectant wipes alcohol canister", 14), ("surface disinfectant 5l", 32), ("hand sanitizer gel 500ml", 8)],
    "Office":   [("printer paper a4 ream 500 sheets", 6), ("toner cartridge black", 75), ("label printer rolls", 12)],
    "Pharma":   [("paracetamol 500mg tablets x100", 11), ("saline solution 0.9% 1l", 4), ("lidocaine 1% vial 20ml", 7)],
}
BRANCHES = ["Tel Aviv", "Haifa", "Beer Sheva", "Jerusalem"]
ABBR = {"gloves": "glv", "box": "bx", "syringe": "syr", "disinfectant": "disinf", "solution": "sol",
        "disposable": "disp", "printer": "prn", "tablets": "tabs", "collection": "coll"}

def messy(text):
    words = text.split()
    words = [ABBR[w] if w in ABBR and random.random() < 0.35 else w for w in words]
    if random.random() < 0.3:                      # typo: drop one letter
        i = random.randrange(len(words))
        if len(words[i]) > 4:
            j = random.randrange(1, len(words[i]) - 1)
            words[i] = words[i][:j] + words[i][j + 1:]
    if random.random() < 0.3:
        words.append(random.choice(["ref", "cat#", "supplier", "lot"]) + " " + str(random.randint(1000, 9999)))
    out = " ".join(words)
    return out.upper() if random.random() < 0.2 else out

def make_lines(n, month_from=1, month_to=6):
    rows = []
    for _ in range(n):
        cat = random.choice(list(CATALOG))
        k = random.randrange(3)
        tmpl, base = CATALOG[cat][k]
        branch = random.choice(BRANCHES)
        unit = base * np.random.uniform(0.95, 1.05)
        if branch == "Haifa" and cat == "PPE":
            unit *= 1.25                            # the planted leak
        qty = random.randint(1, 40)
        rows.append({"month": random.randint(month_from, month_to), "branch": branch,
                     "sku": f"{cat[:3].upper()}-{k+1:02d}",
                     "description": messy(tmpl.format(s=random.choice(["s", "m", "l", "xl"]))),
                     "qty": qty, "unit_price": round(unit, 2), "total": round(unit * qty, 2), "category": cat})
    return pd.DataFrame(rows)

df = make_lines(4000)
df.sample(8, random_state=1)[["branch", "sku", "description", "qty", "total", "category"]]

,branch,sku,description,qty,total,category
200,Haifa,LAB-02,glucose test stris x50,15,395.54,Lab
1078,Tel Aviv,OFF-01,printer paper a4 ream 500 sheets,39,227.41,Office
610,Beer Sheva,SYR-01,syringe 5ml luer lock x100,16,473.15,Syringes
2159,Haifa,PPE-03,isolation gown disp,31,357.43,PPE
1169,Beer Sheva,OFF-03,label printer rolls supplier 3928,17,199.66,Office
2448,Beer Sheva,PHA-01,PARACETAML 500MG TABLETS X100 REF 8519,23,241.81,Pharma
2938,Beer Sheva,CLE-03,hand sanitizer gel 500ml,3,25.09,Cleaning
813,Haifa,PPE-01,nitrile gloves size l bx of 100,18,890.98,PPE


<div dir="rtl">

## 4. מטקסט לטנזור

`EmbeddingBag` לא צריך padding: מקבל רצף אחד ארוך של מזהים ו-`offsets` שמסמנים איפה מתחילה כל שורה.

</div>

In [8]:
classes = sorted(CATALOG)
cls2id = {c: i for i, c in enumerate(classes)}

def to_tensors(texts):
    ids, offsets = [], []
    for t in texts:
        offsets.append(len(ids))
        ids.extend(encode(t))
    return torch.tensor(ids), torch.tensor(offsets)

ids, offsets = to_tensors(df.description[:3])
print("ids:", ids.shape, "offsets:", offsets.tolist())

idx = np.random.permutation(len(df))
train_df, test_df = df.iloc[idx[:2800]], df.iloc[idx[2800:]]
y_train = torch.tensor(train_df.category.map(cls2id).values)
y_test = torch.tensor(test_df.category.map(cls2id).values)

ids: torch.Size([33]) offsets: [0, 14, 24]


<div dir="rtl">

## 5. המודל: EmbeddingBag ← ReLU ← Dropout ← קטגוריה

</div>

In [9]:
class InvoiceClassifier(nn.Module):
    def __init__(self, vocab_size, n_classes, dim=64):
        super().__init__()
        self.emb = nn.EmbeddingBag(vocab_size, dim, mode="mean")
        self.head = nn.Sequential(nn.Linear(dim, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, n_classes))

    def forward(self, ids, offsets):
        return self.head(self.emb(ids, offsets))

model = InvoiceClassifier(len(vocab) + 1, len(classes)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=5e-3)
loss_fn = nn.CrossEntropyLoss()
texts = train_df.description.tolist()

for epoch in range(6):
    model.train()
    perm = torch.randperm(len(texts))
    total = 0.0
    for b in range(0, len(texts), 64):
        bi = perm[b:b + 64]
        ids, offsets = to_tensors([texts[i] for i in bi])
        loss = loss_fn(model(ids.to(device), offsets.to(device)), y_train[bi].to(device))
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(bi)
    print(f"epoch {epoch+1}: loss {total / len(texts):.4f}")

model.eval()
with torch.no_grad():
    ids, offsets = to_tensors(test_df.description)
    pred = model(ids.to(device), offsets.to(device)).argmax(1).cpu()
print(f"\ntest accuracy: {(pred == y_test).float().mean():.3f}")

epoch 1: loss 0.6557
epoch 2: loss 0.0088
epoch 3: loss 0.0044
epoch 4: loss 0.0020
epoch 5: loss 0.0017
epoch 6: loss 0.0011

test accuracy: 1.000


<div dir="rtl">

## 6. איפה הכסף: חודש חדש, בלי קטגוריות

מקבלים חשבוניות של חודש 7 בלי תיוג. המודל מסווג, ואנחנו מסכמים לפי סניף וקטגוריה.

</div>

In [10]:
new = make_lines(800, 7, 7).drop(columns="category")
with torch.no_grad():
    ids, offsets = to_tensors(new.description)
    new["category"] = [classes[i] for i in model(ids.to(device), offsets.to(device)).argmax(1).cpu()]

spend = new.pivot_table(index="category", columns="branch", values="total", aggfunc="sum").round(0)
spend

branch,Beer Sheva,Haifa,Jerusalem,Tel Aviv
category,,,,
Cleaning,16620.0,8380.0,9684.0,11973.0
Lab,55731.0,32188.0,48838.0,32068.0
Office,18313.0,27666.0,19872.0,10839.0
PPE,13047.0,20380.0,16287.0,20230.0
Pharma,6048.0,6423.0,5677.0,5029.0
Syringes,17097.0,12192.0,16545.0,13699.0


In [11]:
# מחיר יחידה ממוצע לכל מק"ט בכל סניף, מול החציון בין הסניפים
price = new.groupby(["sku", "branch"]).unit_price.mean().unstack()
ratio = price.div(price.median(axis=1), axis=0)

flags = ratio.stack().rename("price_vs_median").reset_index()
flags = flags[flags.price_vs_median > 1.10]
qty = new.groupby(["sku", "branch"]).qty.sum().rename("qty").reset_index()
flags = flags.merge(qty, on=["sku", "branch"])
flags["median_price"] = flags.sku.map(price.median(axis=1))
flags["overpaid"] = ((flags.price_vs_median - 1) * flags.median_price * flags.qty).round(0)
flags.sort_values("overpaid", ascending=False)

,sku,branch,price_vs_median,qty,median_price,overpaid
1,PPE-02,Haifa,1.244119,390,21.986818,2093.0
0,PPE-01,Haifa,1.264509,159,37.836818,1591.0
2,PPE-03,Haifa,1.241281,193,8.964750,417.0


<div dir="rtl">

## מה ראינו

- המודל מסווג שורות חשבונית מבולגנות, כולל קיצורים ושגיאות כתיב, כי WordPiece מפרק כל מילה לחלקים מוכרים.
- דיוק של כמעט 100% כאן הוא תוצאה של נתונים סינתטיים פשוטים. על חשבוניות אמיתיות מצפים לפחות, ולכן חשוב סט בדיקה מתויג ידנית.
- הפער שנשתל עלה מעצמו: סניף חיפה, מק״טים של PPE, וסכום עודף לחודש.
- בפרויקט אמיתי: מחליפים את הנתונים הסינתטיים ביצוא מהנהלת החשבונות, ואת המסווג אפשר לשדרג ל-BERT מאומן מראש עם אותו אוצר מילים.

## תרגילים

1. הדפיסו את כל השורות שבהן `tokenize` החזיר `[UNK]`. אילו מילים לא קיימות באוצר המילים, ולמה?
2. הורידו את `Dropout` ובדקו את דיוק הבדיקה. מה קרה?
3. הוסיפו קטגוריה שביעית (למשל `Dental`) עם שלושה פריטים, ואמנו מחדש.

</div>

```markdown
## תרגיל 3: הוספת קטגוריית Dental ואימון מחדש של המודל
```

In [13]:
# 1. נגדיר קטלוג מעודכן הכולל את הקטגוריה החדשה
UPDATED_CATALOG = CATALOG.copy()
UPDATED_CATALOG["Dental"] = [
    ("dental mirror stainless steel", 12),
    ("composite resin filling syringe", 25),
    ("dental floss mint waxed 50m", 5)
]

# 2. נייצר נתונים סינתטיים חדשים הכוללים את קטגוריית Dental
def make_lines_updated(n, month_from=1, month_to=6):
    rows = []
    for _ in range(n):
        cat = random.choice(list(UPDATED_CATALOG))
        k = random.randrange(3)
        tmpl, base = UPDATED_CATALOG[cat][k]
        branch = random.choice(BRANCHES)
        unit = base * np.random.uniform(0.95, 1.05)
        if branch == "Haifa" and cat == "PPE":
            unit *= 1.25
        qty = random.randint(1, 40)
        rows.append({"month": random.randint(month_from, month_to), "branch": branch,
                     "sku": f"{cat[:3].upper()}-{k+1:02d}",
                     "description": messy(tmpl.format(s=random.choice(["s", "m", "l", "xl"]))) if "{s}" in tmpl else messy(tmpl),
                     "qty": qty, "unit_price": round(unit, 2), "total": round(unit * qty, 2), "category": cat})
    return pd.DataFrame(rows)

new_df = make_lines_updated(4500)
print("Dental category samples:")
display(new_df[new_df.category == "Dental"].head(3))

Dental category samples:


,month,branch,sku,description,qty,unit_price,total,category
4,6,Tel Aviv,DEN-03,dental floss mint waxed 50m,20,5.06,101.25,Dental
10,6,Tel Aviv,DEN-02,composite resin filling syr,21,24.74,519.60,Dental
11,3,Haifa,DEN-03,dental flss mint waxed 50m,35,4.87,170.54,Dental


In [14]:
# 3. נכין את הנתונים החדשים לטנזורים
updated_classes = sorted(UPDATED_CATALOG)
updated_cls2id = {c: i for i, c in enumerate(updated_classes)}

idx = np.random.permutation(len(new_df))
train_df_up, test_df_up = new_df.iloc[idx[:3150]], new_df.iloc[idx[3150:]]
y_train_up = torch.tensor(train_df_up.category.map(updated_cls2id).values)
y_test_up = torch.tensor(test_df_up.category.map(updated_cls2id).values)

# 4. נאתחל ונאמן את המודל החדש עם 7 מחלקות
model_up = InvoiceClassifier(len(vocab) + 1, len(updated_classes)).to(device)
opt_up = torch.optim.Adam(model_up.parameters(), lr=5e-3)
loss_fn = nn.CrossEntropyLoss()
texts_up = train_df_up.description.tolist()

for epoch in range(6):
    model_up.train()
    perm = torch.randperm(len(texts_up))
    total = 0.0
    for b in range(0, len(texts_up), 64):
        bi = perm[b:b + 64]
        ids_b, offsets_b = to_tensors([texts_up[i] for i in bi])
        loss = loss_fn(model_up(ids_b.to(device), offsets_b.to(device)), y_train_up[bi].to(device))
        opt_up.zero_grad(); loss.backward(); opt_up.step()
        total += loss.item() * len(bi)
    print(f"epoch {epoch+1}: loss {total / len(texts_up):.4f}")

model_up.eval()
with torch.no_grad():
    ids_t, offsets_t = to_tensors(test_df_up.description)
    pred_up = model_up(ids_t.to(device), offsets_t.to(device)).argmax(1).cpu()
print(f"\nUpdated test accuracy (with 7 classes): {(pred_up == y_test_up).float().mean():.3f}")

epoch 1: loss 0.6581
epoch 2: loss 0.0102
epoch 3: loss 0.0040
epoch 4: loss 0.0024
epoch 5: loss 0.0018
epoch 6: loss 0.0011

Updated test accuracy (with 7 classes): 0.999
